In [4]:
import pandas as pd
from pathlib import Path

bbox_path = Path("metadata/train_bounding_boxes.csv")
manifest_path = Path("metadata/vggface2_selected_129465.csv")

bbox = pd.read_csv(bbox_path)

# Reproducible random selection
selected = (
    bbox.groupby(bbox["NAME_ID"].str.split("/").str[0], group_keys=False)
        .sample(n=15, random_state=42)
        .sort_values("NAME_ID")
        .reset_index(drop=True)
)

selected.to_csv(manifest_path, index=False)

print("Selected images:", len(selected))
print("Selected identities:", selected["NAME_ID"].str.split("/").str[0].nunique())
print("Images per identity:", selected["NAME_ID"].str.split("/").str[0].value_counts().unique())
print("Manifest:", manifest_path)

Selected images: 129465
Selected identities: 8631
Images per identity: [15]
Manifest: metadata\vggface2_selected_129465.csv


In [5]:
import pandas as pd

manifest = pd.read_csv("metadata/vggface2_selected_129465.csv")

identity_counts = (
    manifest["NAME_ID"]
    .str.split("/")
    .str[0]
    .value_counts()
)

print("Total images:", len(manifest))
print("Total identities:", len(identity_counts))
print("Minimum images per identity:", identity_counts.min())
print("Maximum images per identity:", identity_counts.max())
print("Duplicate image IDs:", manifest["NAME_ID"].duplicated().sum())

Total images: 129465
Total identities: 8631
Minimum images per identity: 15
Maximum images per identity: 15
Duplicate image IDs: 0


In [6]:

from pathlib import Path

selected_dir = Path("data/selected")

images = list(selected_dir.rglob("*.jpg"))

print("Extracted JPG files:", len(images))

# Check identity folders
identities = {
    image.parent.name
    for image in images
}

print("Identity folders:", len(identities))

# Check images per identity
counts = {}

for image in images:
    identity = image.parent.name
    counts[identity] = counts.get(identity, 0) + 1

print("Minimum images per identity:", min(counts.values()))
print("Maximum images per identity:", max(counts.values()))
print("Total size (GB):", round(
    sum(image.stat().st_size for image in images) / (1024**3), 2
))

Extracted JPG files: 129465
Identity folders: 8631
Minimum images per identity: 15
Maximum images per identity: 15
Total size (GB): 1.5


In [7]:
from pathlib import Path
from PIL import Image

selected_dir = Path("data/selected")
images = list(selected_dir.rglob("*.jpg"))

bad = []

for i, path in enumerate(images, 1):
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        bad.append((str(path), str(e)))

    if i % 10000 == 0:
        print(f"Checked: {i}/{len(images)}")

print("\nValidation complete.")
print("Total images:", len(images))
print("Bad/unreadable images:", len(bad))

if bad:
    print("\nFirst 5 bad files:")
    for item in bad[:5]:
        print(item)

Checked: 10000/129465
Checked: 20000/129465
Checked: 30000/129465
Checked: 40000/129465
Checked: 50000/129465
Checked: 60000/129465
Checked: 70000/129465
Checked: 80000/129465
Checked: 90000/129465
Checked: 100000/129465
Checked: 110000/129465
Checked: 120000/129465

Validation complete.
Total images: 129465
Bad/unreadable images: 0
